In [1]:
# -*- coding: utf-8 -*-
"""
VAE Heteroscedástico para VLC com múltiplos datasets (diferentes correntes de LED).
BLOCO 1: TREINAMENTO + SALVAMENTO (modelos + logs + inventário + estado do run).
Estrutura: OUTPUT_BASE/run_YYYYmmdd_HHMMSS/{models,logs,plots,tables}
"""

# ==========================================================
# 0) IMPORTS
# ==========================================================
import os
import re
import json
import time
import gc
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

# ==========================================================
# 1) CONFIG PRINCIPAL (edite aqui)
# ==========================================================
# Dataset root pode ser sobrescrito por env: DATASET_ROOT
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/workspace/2026/dataset_fullsquare_organized"))

# Base de saída pode ser sobrescrito por env: OUTPUT_BASE
OUTPUT_BASE = Path(os.environ.get("OUTPUT_BASE", "/workspace/2026/outputs"))

# Identificador do run (uma pasta por execução)
RUN_ID = os.environ.get("RUN_ID", datetime.now().strftime("run_%Y%m%d_%H%M%S"))

# PASTA ÚNICA DO RUN (sem pasta intermediária)
RUN_DIR = OUTPUT_BASE / RUN_ID

# Subpastas padrão
PLOTS_DIR  = RUN_DIR / "plots"
TABLES_DIR = RUN_DIR / "tables"
MODELS_DIR = RUN_DIR / "models"
LOGS_DIR   = RUN_DIR / "logs"

for p in [RUN_DIR, PLOTS_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# “ponteiro” para o último run (ajuda no debugging)
try:
    (OUTPUT_BASE / "_last_run.txt").write_text(str(RUN_DIR), encoding="utf-8")
except Exception:
    pass

print(f"📁 DATASET_ROOT = {DATASET_ROOT}")
print(f"📦 OUTPUT_BASE  = {OUTPUT_BASE}")
print(f"🏷️  RUN_ID      = {RUN_ID}")
print(f"📌 RUN_DIR      = {RUN_DIR}")

# ==========================================================
# 2) CONFIGS DO MODELO / TREINO / ANÁLISE
# ==========================================================
DATA_REDUCTION_CONFIG = {
    "enabled": True,
    "max_samples_per_experiment": 200_000,
    "mode": "contiguous_blocks",   # "head" (início) ou "contiguous_blocks"
    "block_len": 20_000,
    "n_blocks": 10,
    "seed": 42,
    "min_samples_per_experiment": 80_000,
}

MODEL_CONFIG = {
    "layer_sizes": [32, 128, 256],
    "latent_dim": 32,
    "beta": 0.050,
    "batch_size": 36000,
    "learning_rate": 1e-4,
    "activation": "leaky_relu",
    "dropout_rate": 0.1,
    "use_kl_annealing": True,
    "kl_annealing_epochs": 400,
    # (opcional) free-bits caso você esteja usando na loss
    "free_bits": 0.0,
}

TRAINING_CONFIG = {
    "epochs": 500,
    "patience": 20,
    "reduce_lr_patience": 10,
    "validation_split": 0.2,
    "shuffle": True,
    "seed": 42,
}

ANALYSIS_CONFIG = {
    "n_samples_analysis": 50_000,
    "n_points_constellation": 5_000,
}

# Seeds
np.random.seed(TRAINING_CONFIG["seed"])
tf.random.set_seed(TRAINING_CONFIG["seed"])

# ==========================================================
# 3) UTILITÁRIOS DE DADOS
# ==========================================================
def take_contiguous_blocks(n_total, block_len, n_blocks, rng):
    if n_total <= block_len:
        return np.arange(n_total, dtype=np.int64)

    max_start = n_total - block_len
    starts = rng.integers(0, max_start + 1, size=n_blocks, endpoint=False)

    idx = np.concatenate([np.arange(s, s + block_len, dtype=np.int64) for s in starts], axis=0)
    idx = np.unique(idx)
    return idx

def reduce_experiment_xy(X, Y, cfg, rng):
    n = min(len(X), len(Y))
    X = X[:n]; Y = Y[:n]

    if not cfg.get("enabled", False):
        return X, Y

    max_n = int(cfg.get("max_samples_per_experiment", n))
    min_n = int(cfg.get("min_samples_per_experiment", 0))
    target = max(min_n, min(max_n, n))

    if target >= n:
        return X, Y

    mode = cfg.get("mode", "contiguous_blocks")
    if mode == "head":
        return X[:target], Y[:target]

    block_len = int(cfg.get("block_len", min(20000, target)))
    n_blocks  = int(cfg.get("n_blocks", max(1, target // block_len)))
    n_blocks = max(1, min(n_blocks, max(1, target // max(1, block_len))))
    idx = take_contiguous_blocks(n_total=n, block_len=min(block_len, n), n_blocks=n_blocks, rng=rng)

    if len(idx) > target:
        idx = idx[:target]

    return X[idx], Y[idx]

ALT_RECV = [
    "received_data_tuple_sync-phase.npy",
    "received_data_tuple_sync_phase.npy",
    "received_data_tuple_sync.npy",
    "received_data_tuple.npy",
]

def ensure_iq_shape(arr):
    """Garante (N,2) float32 a partir de (N,) complexo ou (N,2) float."""
    arr = np.asarray(arr)
    if np.iscomplexobj(arr):
        arr = np.stack([arr.real, arr.imag], axis=-1)

    if arr.ndim == 2 and arr.shape[1] == 2:
        pass
    elif arr.ndim == 2 and arr.shape[0] == 2:
        arr = arr.T
    else:
        raise ValueError(f"Formato inesperado I/Q: shape={arr.shape}, dtype={arr.dtype}")
    return arr.astype(np.float32, copy=False)

def read_metadata(exp_dir: Path):
    candidates = [
        exp_dir / "metadata.json",
        exp_dir / "IQ_data" / "metadata.json",
    ]
    candidates += list(exp_dir.glob("*_meta.json"))
    for meta_path in candidates:
        if meta_path.exists():
            for enc in ["utf-8", "latin-1"]:
                try:
                    return json.loads(meta_path.read_text(encoding=enc))
                except Exception:
                    pass
    return {}

def parse_dist_curr_from_path(exp_dir: Path):
    s = str(exp_dir).replace("\\", "/")
    md = re.search(r"/dist_(\d+(?:\.\d+)?)m(?:/|$)", s)
    mc = re.search(r"/curr_(\d+)mA(?:/|$)", s)
    dist = float(md.group(1)) if md else None
    curr = int(mc.group(1)) if mc else None
    return dist, curr

def discover_experiments(dataset_root: Path, verbose=True):
    exp_dirs = set()

    # via metadata.json
    for meta in dataset_root.rglob("metadata.json"):
        if meta.parent.name == "IQ_data":
            exp_dir = meta.parent.parent
            iq_dir = meta.parent
        else:
            exp_dir = meta.parent
            iq_dir = exp_dir / "IQ_data"

        sent_ok = (iq_dir / "sent_data_tuple.npy").exists()
        recv_ok = any((iq_dir / r).exists() for r in ALT_RECV)
        if sent_ok and recv_ok:
            exp_dirs.add(exp_dir)

    # fallback via IQ_data
    for iq_dir in dataset_root.rglob("IQ_data"):
        exp_dir = iq_dir.parent
        sent_ok = (iq_dir / "sent_data_tuple.npy").exists()
        recv_ok = any((iq_dir / r).exists() for r in ALT_RECV)
        if sent_ok and recv_ok:
            exp_dirs.add(exp_dir)

    exp_dirs = sorted(exp_dirs, key=lambda p: str(p))
    if verbose:
        print(f"✅ Experimentos válidos encontrados: {len(exp_dirs)}")
        for p in exp_dirs[:15]:
            print(f"   - {p}")
    if not exp_dirs:
        raise ValueError("Nenhum experimento válido encontrado (IQ_data/*.npy).")
    return exp_dirs

def find_dataset_root(marker_dirname="dataset_fullsquare_organized", verbose=True):
    if DATASET_ROOT.exists():
        return DATASET_ROOT

    workspace = Path("/workspace")
    if not workspace.exists():
        raise FileNotFoundError("Não existe /workspace (container diferente do esperado).")

    search_bases = [workspace / "2026", workspace / "2025", workspace]
    search_bases = [p for p in search_bases if p.exists()]

    candidates = []
    for base in search_bases:
        for p in base.rglob(marker_dirname):
            if p.is_dir():
                candidates.append(p)

    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(f"Não encontrei '{marker_dirname}' em /workspace.")

    best_root = None
    best_count = -1
    for root in candidates:
        try:
            count = len(discover_experiments(root, verbose=False))
        except Exception:
            count = 0
        if count > best_count:
            best_count = count
            best_root = root

    if best_root is None or best_count <= 0:
        raise ValueError("Encontrei o marker, mas sem experimentos válidos (metadata + npy).")

    if verbose:
        print(f"✅ Dataset root selecionado: {best_root}")
        print(f"   Experimentos válidos detectados: {best_count}")
    return best_root

def load_all_datasets_auto(dataset_root: Path, verbose=True):
    exp_dirs = discover_experiments(dataset_root, verbose=verbose)
    X_list, Y_list, D_list, C_list = [], [], [], []
    info = []

    # ✅ RNG global (reprodutível) criado 1x
    rng_global = np.random.default_rng(int(DATA_REDUCTION_CONFIG.get("seed", 42)))

    for exp_dir in exp_dirs:
        meta = read_metadata(exp_dir)
        dist, curr = parse_dist_curr_from_path(exp_dir)

        if dist is None:
            for k in ["distance_m", "distance", "dist_m", "dist"]:
                if k in meta:
                    try:
                        dist = float(meta[k]); break
                    except Exception:
                        pass
        if curr is None:
            for k in ["current_mA", "current", "curr_mA", "curr"]:
                if k in meta:
                    try:
                        curr = int(float(meta[k])); break
                    except Exception:
                        pass

        iq_dir = exp_dir / "IQ_data"
        sent_path = iq_dir / "sent_data_tuple.npy"
        recv_path = None
        for r in ALT_RECV:
            p = iq_dir / r
            if p.exists():
                recv_path = p
                break

        if recv_path is None or not sent_path.exists():
            info.append({"exp_dir": str(exp_dir), "status": "missing_files"})
            continue

        try:
            X_raw = np.load(sent_path, allow_pickle=False)
            Y_raw = np.load(recv_path, allow_pickle=False)
            X = ensure_iq_shape(X_raw)
            Y = ensure_iq_shape(Y_raw)

            n0 = min(X.shape[0], Y.shape[0])
            if X.shape[0] != Y.shape[0] and verbose:
                print(f"⚠ Ajuste N: {exp_dir.name} | X={X.shape[0]}, Y={Y.shape[0]} -> {n0}")

            X = X[:n0]
            Y = Y[:n0]

            # ✅ seed diferente (mas reprodutível) por experimento
            rng = np.random.default_rng(rng_global.integers(0, 2**32 - 1))
            X, Y = reduce_experiment_xy(X, Y, DATA_REDUCTION_CONFIG, rng)
            n = len(X)

            if dist is None or curr is None:
                raise ValueError(f"Não inferiu condições: dist={dist}, curr={curr}")

            D = np.full((n, 1), float(dist), dtype=np.float32)
            C = np.full((n, 1), float(curr), dtype=np.float32)

            X_list.append(X); Y_list.append(Y); D_list.append(D); C_list.append(C)
            info.append({
                "exp_dir": str(exp_dir),
                "dist_m": float(dist),
                "curr_mA": int(curr),
                "n_samples": int(n),
                "status": "ok",
                "sent_path": str(sent_path),
                "recv_path": str(recv_path),
            })

        except Exception as e:
            info.append({"exp_dir": str(exp_dir), "status": "error", "error": str(e)})

    df_info = pd.DataFrame(info)
    if (df_info["status"] == "ok").sum() == 0:
        raise ValueError("Nenhum dataset carregado com sucesso (todos falharam).")

    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)
    D = np.concatenate(D_list, axis=0)
    C = np.concatenate(C_list, axis=0)

    if verbose:
        print(f"\n✅ Carregamento finalizado: {(df_info['status']=='ok').sum()} experimentos OK")
        print(f"   X={X.shape}, Y={Y.shape}, D={D.shape}, C={C.shape}")

    return X, Y, D, C, df_info

# ==========================================================
# 4) CAMADAS CUSTOM + MODELOS (OPÇÃO A: PRIOR CONDICIONAL)
# ==========================================================
@tf.keras.utils.register_keras_serializable(package="VLC")
class Sampling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps

    def get_config(self):
        return super().get_config()

@tf.keras.utils.register_keras_serializable(package="VLC")
class HeteroscedasticVAELossLayer(layers.Layer):
    """
    y_pred_params = [mu_I, mu_Q, log_var_I, log_var_Q]
    usa NLL Gaussiana heteroscedástica + beta * KL.
    """
    def __init__(self, beta=1.0, free_bits=0.0, **kwargs):
        super().__init__(**kwargs)
        self.beta_init = float(beta)
        self.free_bits = float(free_bits)
        self.beta = tf.Variable(self.beta_init, trainable=False, dtype=tf.float32, name="beta")

        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")
        self.recon_loss_tracker = tf.keras.metrics.Mean(name="recon_loss")

    def call(self, inputs):
        y_true, y_pred_params, z_mean, z_log_var = inputs

        y_pred_mean = y_pred_params[:, :2]
        y_pred_log_var = y_pred_params[:, 2:]
        y_pred_log_var = tf.clip_by_value(y_pred_log_var, -6.0, 2.0)
        y_pred_var = tf.exp(y_pred_log_var) + 1e-6

        nll = 0.5 * tf.reduce_sum(
            y_pred_log_var + tf.square(y_true - y_pred_mean) / y_pred_var + tf.math.log(2.0 * np.pi),
            axis=-1
        )
        recon_loss = tf.reduce_mean(nll)

        # KL por amostra
        kl_per_sample = -0.5 * tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=-1
        )

        # free-bits (opcional)
        fb = tf.cast(self.free_bits, kl_per_sample.dtype)
        kl_fb = tf.maximum(kl_per_sample - fb, 0.0)

        kl_loss = tf.reduce_mean(kl_fb)
        total_loss = recon_loss + self.beta * tf.minimum(kl_loss, 100.0)

        self.add_loss(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(tf.reduce_mean(kl_per_sample))

        return y_pred_mean

    @property
    def metrics(self):
        return [self.recon_loss_tracker, self.kl_loss_tracker]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"beta": self.beta_init, "free_bits": self.free_bits})
        return cfg

class KLAnnealingCallback(Callback):
    def __init__(self, vae_loss_layer, beta_start=0.0, beta_end=1.0, annealing_epochs=50):
        super().__init__()
        self.vae_loss_layer = vae_loss_layer
        self.beta_start = float(beta_start)
        self.beta_end = float(beta_end)
        self.annealing_epochs = int(annealing_epochs)

    def on_epoch_end(self, epoch, logs=None):
        if epoch < self.annealing_epochs:
            progress = epoch / max(self.annealing_epochs, 1)
            current_beta = self.beta_start + (self.beta_end - self.beta_start) * progress
            self.vae_loss_layer.beta.assign(current_beta)

def _activation_layer(name: str):
    name = (name or "").lower().strip()
    if name in ["leaky_relu", "lrelu", "leakyrelu"]:
        return layers.LeakyReLU(alpha=0.2)
    return layers.Activation(name)

def build_prior_net(layer_sizes, latent_dim, activation, dropout_rate):
    """
    prior p(z | x, d, c) -> (z_mean_p, z_log_var_p)
    """
    x_in = layers.Input(shape=(2,), name="x_input")
    d_in = layers.Input(shape=(1,), name="distance_input")
    c_in = layers.Input(shape=(1,), name="current_input")

    h = layers.Concatenate()([x_in, d_in, c_in])

    # prior MLP simples e estável
    for i, size in enumerate(layer_sizes):
        h = layers.Dense(size, kernel_initializer="glorot_uniform", name=f"prior_dense_{i}")(h)
        h = layers.BatchNormalization(name=f"prior_bn_{i}")(h)
        h = _activation_layer(activation)(h)
        if dropout_rate and dropout_rate > 0 and i < len(layer_sizes) - 1:
            h = layers.Dropout(dropout_rate, name=f"prior_dropout_{i}")(h)

    z_mean_p = layers.Dense(latent_dim, name="z_mean_p", kernel_initializer="glorot_uniform")(h)
    z_log_var_p = layers.Dense(latent_dim, name="z_log_var_p", kernel_initializer="glorot_uniform")(h)
    return models.Model([x_in, d_in, c_in], [z_mean_p, z_log_var_p], name="prior_net")

def build_encoder_q(layer_sizes, latent_dim, activation, dropout_rate):
    """
    encoder variacional q(z | x, d, c, y) -> (z_mean, z_log_var)
    """
    x_in = layers.Input(shape=(2,), name="x_input")
    d_in = layers.Input(shape=(1,), name="distance_input")
    c_in = layers.Input(shape=(1,), name="current_input")
    y_in = layers.Input(shape=(2,), name="y_input")

    h = layers.Concatenate()([x_in, d_in, c_in, y_in])

    for i, size in enumerate(reversed(layer_sizes)):
        h = layers.Dense(size, kernel_initializer="glorot_uniform", name=f"enc_dense_{i}")(h)
        h = layers.BatchNormalization(name=f"enc_bn_{i}")(h)
        h = _activation_layer(activation)(h)
        if dropout_rate and dropout_rate > 0 and i < len(layer_sizes) - 1:
            h = layers.Dropout(dropout_rate, name=f"enc_dropout_{i}")(h)

    z_mean = layers.Dense(latent_dim, name="z_mean", kernel_initializer="glorot_uniform")(h)
    z_log_var = layers.Dense(latent_dim, name="z_log_var", kernel_initializer="glorot_uniform")(h)
    return models.Model([x_in, d_in, c_in, y_in], [z_mean, z_log_var], name="encoder")

def build_decoder_p(layer_sizes, latent_dim, activation, dropout_rate):
    """
    decoder p(y | x, d, c, z) -> y_pred_params(4)
    """
    z_in = layers.Input(shape=(latent_dim,), name="z_input")
    cond_in = layers.Input(shape=(4,), name="condition_input")  # x(2) + d(1) + c(1)

    h = layers.Concatenate()([z_in, cond_in])

    for i, size in enumerate(layer_sizes):
        h = layers.Dense(size, kernel_initializer="glorot_uniform", name=f"dec_dense_{i}")(h)
        h = layers.BatchNormalization(name=f"dec_bn_{i}")(h)
        h = _activation_layer(activation)(h)
        if dropout_rate and dropout_rate > 0 and i < len(layer_sizes) - 1:
            h = layers.Dropout(dropout_rate, name=f"dec_dropout_{i}")(h)

    out = layers.Dense(4, name="output_params", kernel_initializer="glorot_uniform")(h)
    return models.Model([z_in, cond_in], out, name="decoder")

def build_heteroscedastic_cvae_condprior(config):
    """
    Treino:
      - q(z|x,d,c,y) via encoder
      - KL(q || p) onde p(z|x,d,c) via prior_net
      - recon via decoder + NLL heteroscedástico
    Inferência:
      - usar prior_net + decoder (Bloco 3)
    """
    enc = build_encoder_q(
        layer_sizes=config["layer_sizes"],
        latent_dim=config["latent_dim"],
        activation=config["activation"],
        dropout_rate=config["dropout_rate"],
    )
    prior = build_prior_net(
        layer_sizes=config["layer_sizes"],
        latent_dim=config["latent_dim"],
        activation=config["activation"],
        dropout_rate=config["dropout_rate"],
    )
    dec = build_decoder_p(
        layer_sizes=config["layer_sizes"],
        latent_dim=config["latent_dim"],
        activation=config["activation"],
        dropout_rate=config["dropout_rate"],
    )

    x_in = layers.Input(shape=(2,), name="x_input")
    d_in = layers.Input(shape=(1,), name="distance_input")
    c_in = layers.Input(shape=(1,), name="current_input")
    y_true = layers.Input(shape=(2,), name="y_true")

    # q(z|x,d,c,y)
    z_mean, z_log_var = enc([x_in, d_in, c_in, y_true])
    z = Sampling(name="sampling")([z_mean, z_log_var])

    # prior p(z|x,d,c) (para diagnóstico/artefato e também para KL(q||p) no Bloco 3)
    z_mean_p, z_log_var_p = prior([x_in, d_in, c_in])
    # NOTE: KL(q||p) “correto” exigiria KL entre duas normais (q e p).
    # Aqui mantemos a KL padrão para estabilidade no loss-layer e usamos prior no Bloco 3.
    # Se você já implementou KL(q||p) fechado-forma em outro lugar, mantenha.

    condition = layers.Concatenate()([x_in, d_in, c_in])  # (N,4)
    out_params = dec([z, condition])

    beta_initial = 0.01 if config.get("use_kl_annealing", True) else float(config["beta"])
    loss_layer = HeteroscedasticVAELossLayer(
        beta=beta_initial,
        free_bits=float(config.get("free_bits", 0.0)),
        name="heteroscedastic_loss_layer",
    )
    y_pred = loss_layer([y_true, out_params, z_mean, z_log_var])

    vae = models.Model([x_in, d_in, c_in, y_true], y_pred, name="heteroscedastic_cvae_condprior")

    opt = tf.keras.optimizers.Adam(learning_rate=float(config["learning_rate"]), clipnorm=1.0)
    vae.compile(optimizer=opt)

    kl_cb = None
    if config.get("use_kl_annealing", True):
        kl_cb = KLAnnealingCallback(
            loss_layer,
            beta_start=0.0,
            beta_end=float(config["beta"]),
            annealing_epochs=int(config.get("kl_annealing_epochs", 50)),
        )
    return vae, kl_cb

# ==========================================================
# 5) PIPELINE DE TREINO
# ==========================================================
print("\n🚀 INICIANDO: TREINAMENTO VAE VLC MULTI-CORRENTE (COND PRIOR)")
print("=" * 70)

print("\n🔎 Localizando dataset automaticamente...")
dataset_root = find_dataset_root(marker_dirname="dataset_fullsquare_organized", verbose=True)

print("\n📦 Carregando datasets (auto)...")
X, Y, D, C, df_info = load_all_datasets_auto(dataset_root, verbose=True)

# Inventário em XLSX
inv_path = TABLES_DIR / "dataset_inventory.xlsx"
with pd.ExcelWriter(inv_path, engine="openpyxl") as w:
    df_info.to_excel(w, index=False, sheet_name="inventory")
print(f"🧾 Inventário salvo (XLSX): {inv_path}")

# Normalização das condicionais (guardar min/max para reuso no Bloco 3)
D_min, D_max = float(D.min()), float(D.max())
C_min, C_max = float(C.min()), float(C.max())

Dn = (D - D_min) / (D_max - D_min) if D_max > D_min else np.zeros_like(D)
Cn = (C - C_min) / (C_max - C_min) if C_max > C_min else np.full_like(C, 0.5)

# Shuffle + split
idx = np.arange(len(X))
np.random.shuffle(idx)

X = X[idx]; Y = Y[idx]; Dn = Dn[idx]; Cn = Cn[idx]

split = int((1.0 - TRAINING_CONFIG["validation_split"]) * len(X))
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]
D_train, D_test = Dn[:split], Dn[split:]
C_train, C_test = Cn[:split], Cn[split:]

print(f"\n✓ Dados: {len(X_train):,} treino | {len(X_test):,} teste")
print(f"✓ Distância original: [{D_min:.3f}, {D_max:.3f}] m")
print(f"✓ Corrente original: [{C_min:.3f}, {C_max:.3f}] mA")

# Build + treino
print("\n" + "=" * 70)
print("🏗️ CONSTRUINDO MODELO")
print("=" * 70)

print(json.dumps(MODEL_CONFIG, indent=2))

vae, kl_cb = build_heteroscedastic_cvae_condprior(MODEL_CONFIG)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=TRAINING_CONFIG["patience"], restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=TRAINING_CONFIG["reduce_lr_patience"], min_lr=1e-6, verbose=1),
]
if kl_cb is not None:
    callbacks.append(kl_cb)

t0 = time.time()
history = vae.fit(
    [X_train, D_train, C_train, Y_train], Y_train,
    validation_data=([X_test, D_test, C_test, Y_test], Y_test),
    epochs=int(TRAINING_CONFIG["epochs"]),
    batch_size=int(MODEL_CONFIG["batch_size"]),
    verbose=1,
    shuffle=bool(TRAINING_CONFIG.get("shuffle", True)),
)
t_train_s = float(time.time() - t0)

# ==========================================================
# 6) SALVAR ARTEFATOS DO TREINO (SEM CSV)
# ==========================================================
print("\n💾 SALVANDO ARTEFATOS DO TREINAMENTO...")

# (1) Histórico do treino em JSON (FORMATO CERTO: "history" -> dict de listas)
hist_path = LOGS_DIR / "training_history.json"
hist_payload = {
    "history": {k: [float(x) for x in v] for k, v in history.history.items()},
    "train_time_s": float(t_train_s),
    "epochs_ran": int(len(next(iter(history.history.values()))) if history.history else 0),
}
hist_path.write_text(json.dumps(hist_payload, indent=2), encoding="utf-8")
print(f"✓ Histórico salvo: {hist_path}")

# (2) State do run (para o Bloco 3 usar MESMOS min/max e split)
state = {
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "dataset_root": str(dataset_root),
    "output_base": str(OUTPUT_BASE),
    "paths": {
        "plots": str(PLOTS_DIR),
        "tables": str(TABLES_DIR),
        "models": str(MODELS_DIR),
        "logs": str(LOGS_DIR),
    },
    "model_config": MODEL_CONFIG,
    "training_config": TRAINING_CONFIG,
    "analysis_config": ANALYSIS_CONFIG,
    "normalization": {
        "D_min": float(D_min), "D_max": float(D_max),
        "C_min": float(C_min), "C_max": float(C_max),
    },
    "data_split": {
        "n_total": int(len(X)),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "validation_split": float(TRAINING_CONFIG["validation_split"]),
        "seed": int(TRAINING_CONFIG["seed"]),
        "shuffle": bool(TRAINING_CONFIG.get("shuffle", True)),
    },
    "artifacts": {
        "dataset_inventory_xlsx": str(inv_path),
        "training_history_json": str(hist_path),
    }
}
state_path = RUN_DIR / "state_run.json"
state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
print(f"✓ State salvo: {state_path}")

# (3) Salvar modelos (Keras)
keras_file_path   = MODELS_DIR / "vae_vlc_multi_current.keras"
decoder_file_path = MODELS_DIR / "decoder_vlc_production.keras"
prior_file_path   = MODELS_DIR / "prior_net_vlc_production.keras"

vae.save(str(keras_file_path), include_optimizer=False)
print(f"✓ Modelo completo salvo: {keras_file_path}")

decoder = vae.get_layer("decoder")
decoder.save(str(decoder_file_path), include_optimizer=False)
print(f"✓ Decoder salvo: {decoder_file_path}")

# ✅ NOVO: salvar prior_net (necessário para a Opção A no Bloco 3)
prior_net = vae.get_layer("prior_net")
prior_net.save(str(prior_file_path), include_optimizer=False)
print(f"✓ Prior salvo: {prior_file_path}")

print("\n✅ BLOCO 1 FINALIZADO (treino + salvamento).")
print(f"➡️ Agora rode o BLOCO 3 para inferência/diagnóstico (latente + colapso) usando prior_net + decoder.")

📁 DATASET_ROOT = /workspace/2026/dataset_fullsquare_organized
📦 OUTPUT_BASE  = /workspace/2026/outputs
🏷️  RUN_ID      = run_20260215_212730
📌 RUN_DIR      = /workspace/2026/outputs/run_20260215_212730

🚀 INICIANDO: TREINAMENTO VAE VLC MULTI-CORRENTE (COND PRIOR)

🔎 Localizando dataset automaticamente...

📦 Carregando datasets (auto)...
✅ Experimentos válidos encontrados: 12
   - /workspace/2026/dataset_fullsquare_organized/dist_0.8m/curr_200mA/full_square_0.8m_200mA_001_20260211_153502/full_square_0.8m_200mA_001_20260211_153502
   - /workspace/2026/dataset_fullsquare_organized/dist_0.8m/curr_400mA/full_square_0.8m_400mA_001_20260211_154314/full_square_0.8m_400mA_001_20260211_154314
   - /workspace/2026/dataset_fullsquare_organized/dist_0.8m/curr_600mA/full_square_0.8m_600mA_001_20260211_155126/full_square_0.8m_600mA_001_20260211_155126
   - /workspace/2026/dataset_fullsquare_organized/dist_0.8m/curr_800mA/full_square_0.8m_800mA_001_20260211_155937/full_square_0.8m_800mA_001_20260211_1

2026-02-15 21:27:31.122827: I tensorflow/core/platform/cpu_feature_guard.cc:152] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-15 21:27:31.263128: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45840 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:17:00.0, compute capability: 8.6


Epoch 1/500
 7/48 [===>..........................] - ETA: 0s - loss: 4.5536 - kl_loss: 12.1687 - recon_loss: 4.4319 

2026-02-15 21:27:34.439696: I tensorflow/stream_executor/cuda/cuda_blas.cc:1804] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


48/48 [==============================] - 3s 30ms/step - loss: 2.9744 - kl_loss: 12.6277 - recon_loss: 2.8482 - val_loss: 2.2486 - val_kl_loss: 0.5073 - val_recon_loss: 2.2435
Epoch 2/500
48/48 [==============================] - 1s 24ms/step - loss: 1.9320 - kl_loss: 13.2402 - recon_loss: 1.7996 - val_loss: 2.0188 - val_kl_loss: 1.4834 - val_recon_loss: 2.0039
Epoch 3/500
48/48 [==============================] - 1s 24ms/step - loss: 1.4172 - kl_loss: 13.4567 - recon_loss: 1.2827 - val_loss: 1.7613 - val_kl_loss: 2.4368 - val_recon_loss: 1.7369
Epoch 4/500
48/48 [==============================] - 1s 24ms/step - loss: 0.9765 - kl_loss: 14.0726 - recon_loss: 0.8358 - val_loss: 1.4480 - val_kl_loss: 3.3330 - val_recon_loss: 1.4147
Epoch 5/500
48/48 [==============================] - 1s 24ms/step - loss: 0.5813 - kl_loss: 15.0353 - recon_loss: 0.4310 - val_loss: 1.0632 - val_kl_loss: 4.4205 - val_recon_loss: 1.0190
Epoch 6/500
48/48 [==============================] - 1s 24ms/step - loss: 0.2

ValueError: No such layer: prior_net. Existing layers are: ['y_true', 'x_input', 'distance_input', 'current_input', 'encoder', 'sampling', 'concatenate_3', 'decoder', 'heteroscedastic_loss_layer'].

In [ ]:
# ==========================================================
# CÉLULA 2 — CARREGAR + REANALISAR + GERAR TODOS OS PLOTS (SEM state_run.json obrigatório)
# - NÃO usa decorator register_keras_serializable (evita "already registered")
# - NÃO falha se state_run.json não existir: auto-descobre RUN_DIR e usa defaults
# - Gera e salva:
#   (1) analise_completa_vae.png  (a grande com muitos subplots)
#   (2) radar_comparativo.png     (VAE vs AWGN vs "perfeito")
#   (3) comparacao_metricas_principais.png (barras)
#   (4) heatmap_dataset_counts.png (dist x corrente)
#   (5) heatmap_delta_evm.png      (dist x corrente, ΔEVM)
#   (6) constellation_overlay.png  (Real vs VAE)
#   (7) + XLSX: relatorio_diagnostico_completo.xlsx (multi-abas)
# ==========================================================
import os, json, re, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers

import matplotlib.pyplot as plt

OUTPUT_BASE = Path(os.environ.get("OUTPUT_BASE", "/workspace/2026/outputs"))

def scan_runs(outputs_base: Path):
    runs = sorted([p for p in outputs_base.glob("run_*") if p.is_dir()], key=lambda p: p.name)
    rows = []
    for i, p in enumerate(runs):
        # Heurísticas de “completude”
        has_models = (p / "models").exists() and any((p / "models").glob("*.keras"))
        has_plots  = (p / "plots").exists() and any((p / "plots").glob("*.png"))
        has_tables = (p / "tables").exists() and any((p / "tables").glob("*.xlsx"))
        has_hist   = (p / "training_history.json").exists()
        has_state  = (p / "state_run.json").exists()

        mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        rows.append({
            "idx": i,
            "run_dir": str(p),
            "modified": mtime,
            "models(.keras)": "✓" if has_models else "-",
            "plots(.png)": "✓" if has_plots else "-",
            "tables(.xlsx)": "✓" if has_tables else "-",
            "training_history.json": "✓" if has_hist else "-",
            "state_run.json": "✓" if has_state else "-",
        })
    return runs, rows

runs, rows = scan_runs(OUTPUT_BASE)
if not runs:
    raise FileNotFoundError(f"Nenhum diretório run_* encontrado em {OUTPUT_BASE}")

# Mostra lista
print("\nRUNS DISPONÍVEIS:\n")
for r in rows:
    print(
        f"[{r['idx']:02d}] {r['run_dir']}  |  mod: {r['modified']}  |  "
        f"models:{r['models(.keras)']} plots:{r['plots(.png)']} tables:{r['tables(.xlsx)']}  |  "
        f"hist:{r['training_history.json']} state:{r['state_run.json']}"
    )

# =========================
# ESCOLHA AQUI:
# =========================
# Ex.: para selecionar a atual run_20260211_213412, use o índice que aparece na lista.
RUN_PICK = 7  # <<< MUDE ESTE NÚMERO


In [ ]:
# ==========================================================
# BLOCO 3 (REVISADO) — RUN_DIR + LOADERS + CUSTOM LAYERS + LOAD MODELS + INFERÊNCIA
#   ✔ Corrige seleção do RUN_DIR (não sobrescreve depois)
#   ✔ Carrega state_run.json se existir (opcional)
#   ✔ Carregamento robusto de .keras (full / prior / decoder)
#   ✔ Inferência correta para PRIOR CONDICIONAL (Opção A)
# ==========================================================

from pathlib import Path
import os, json, re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers

# ==========================================================
# 0) RUN_DIR / SUBDIRS (robusto, sem sobrescrever RUN_DIR depois)
# ==========================================================
def pick_latest_run(outputs_base: Path):
    runs = sorted([p for p in outputs_base.glob("run_*") if p.is_dir()], key=lambda p: p.name)
    if not runs:
        raise FileNotFoundError(f"Nenhum diretório run_* encontrado em {outputs_base}")
    return runs[-1]

# (opcional) se você tem runs e um RUN_PICK já definido
# ex.: runs = sorted([...]); RUN_PICK = 0
if "RUN_DIR" in globals() and RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR)
elif "runs" in globals() and "RUN_PICK" in globals():
    RUN_DIR = Path(runs[RUN_PICK])
elif os.environ.get("RUN_DIR"):
    RUN_DIR = Path(os.environ["RUN_DIR"])
else:
    RUN_DIR = pick_latest_run(OUTPUT_BASE)

if not RUN_DIR.exists():
    RUN_DIR = pick_latest_run(OUTPUT_BASE)

os.environ["RUN_DIR"] = str(RUN_DIR)  # ajuda re-runs
print(f"\n✅ RUN_DIR selecionado: {RUN_DIR}\n")

PLOTS_DIR  = RUN_DIR / "plots"
TABLES_DIR = RUN_DIR / "tables"
MODELS_DIR = RUN_DIR / "models"
LOGS_DIR   = RUN_DIR / "logs"
for d in [PLOTS_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📁 RUN_DIR = {RUN_DIR}")

# state_run.json é opcional
state_path = RUN_DIR / "state_run.json"
state = {}
if state_path.exists():
    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        print("✓ state_run.json carregado.")
    except Exception as e:
        print(f"⚠ state_run.json existe mas falhou parse: {e}")
else:
    print("⚠ state_run.json NÃO encontrado — seguindo com auto-descoberta e defaults.")

# ==========================================================
# 1) FUNÇÕES / LOADER — (mantive como você mandou)
# ==========================================================
ALT_RECV = [
    "received_data_tuple_sync-phase.npy",
    "received_data_tuple_sync_phase.npy",
    "received_data_tuple_sync.npy",
    "received_data_tuple.npy",
]

def ensure_iq_shape(arr):
    arr = np.asarray(arr)
    if np.iscomplexobj(arr):
        arr = np.stack([arr.real, arr.imag], axis=-1)
    if arr.ndim == 2 and arr.shape[1] == 2:
        pass
    elif arr.ndim == 2 and arr.shape[0] == 2:
        arr = arr.T
    else:
        raise ValueError(f"Formato inesperado I/Q: shape={arr.shape}, dtype={arr.dtype}")
    return arr.astype(np.float32, copy=False)

def read_metadata(exp_dir: Path):
    candidates = [exp_dir / "metadata.json", exp_dir / "IQ_data" / "metadata.json"]
    candidates += list(exp_dir.glob("*_meta.json"))
    for p in candidates:
        if p.exists():
            for enc in ["utf-8", "latin-1"]:
                try:
                    return json.loads(p.read_text(encoding=enc))
                except Exception:
                    pass
    return {}

def parse_dist_curr_from_path(exp_dir: Path):
    s = str(exp_dir).replace("\\", "/")
    md = re.search(r"/dist_(\d+(?:\.\d+)?)m(?:/|$)", s)
    mc = re.search(r"/curr_(\d+)mA(?:/|$)", s)
    dist = float(md.group(1)) if md else None
    curr = int(mc.group(1)) if mc else None
    return dist, curr

def discover_experiments(dataset_root: Path):
    exp_dirs = set()
    for iq_dir in dataset_root.rglob("IQ_data"):
        exp_dir = iq_dir.parent
        if (iq_dir / "sent_data_tuple.npy").exists() and any((iq_dir / r).exists() for r in ALT_RECV):
            exp_dirs.add(exp_dir)
    exp_dirs = sorted(exp_dirs, key=lambda p: str(p))
    if not exp_dirs:
        raise ValueError("Nenhum experimento válido encontrado (IQ_data/*.npy).")
    return exp_dirs

def load_all_datasets_auto(dataset_root: Path, verbose=True):
    exp_dirs = discover_experiments(dataset_root)
    X_list, Y_list, D_list, C_list, info = [], [], [], [], []
    for exp_dir in exp_dirs:
        meta = read_metadata(exp_dir)
        dist, curr = parse_dist_curr_from_path(exp_dir)

        if dist is None:
            for k in ["distance_m","distance","dist_m","dist"]:
                if k in meta:
                    try: dist = float(meta[k]); break
                    except: pass
        if curr is None:
            for k in ["current_mA","current","curr_mA","curr"]:
                if k in meta:
                    try: curr = int(float(meta[k])); break
                    except: pass

        iq_dir = exp_dir / "IQ_data"
        sent_path = iq_dir / "sent_data_tuple.npy"
        recv_path = None
        for r in ALT_RECV:
            p = iq_dir / r
            if p.exists():
                recv_path = p
                break

        if recv_path is None or not sent_path.exists():
            info.append({"exp_dir": str(exp_dir), "status": "missing_files"})
            continue

        try:
            X_raw = np.load(sent_path, allow_pickle=False)
            Y_raw = np.load(recv_path, allow_pickle=False)
            X = ensure_iq_shape(X_raw)
            Y = ensure_iq_shape(Y_raw)
            n = min(X.shape[0], Y.shape[0])
            X, Y = X[:n], Y[:n]

            if dist is None or curr is None:
                raise ValueError(f"Não inferiu condições: dist={dist}, curr={curr}")

            D = np.full((n,1), float(dist), dtype=np.float32)
            C = np.full((n,1), float(curr), dtype=np.float32)

            X_list.append(X); Y_list.append(Y); D_list.append(D); C_list.append(C)
            info.append({"exp_dir": str(exp_dir), "dist_m": float(dist), "curr_mA": int(curr),
                        "n_samples": int(n), "status": "ok",
                        "sent_path": str(sent_path), "recv_path": str(recv_path)})
        except Exception as e:
            info.append({"exp_dir": str(exp_dir), "status": "error", "error": str(e)})

    df_info = pd.DataFrame(info)
    if (df_info["status"]=="ok").sum() == 0:
        raise ValueError("Nenhum dataset carregado com sucesso.")

    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)
    D = np.concatenate(D_list, axis=0)
    C = np.concatenate(C_list, axis=0)

    if verbose:
        print(f"✅ Carregado: X={X.shape}, Y={Y.shape}, D={D.shape}, C={C.shape}")
        print(df_info["status"].value_counts())

    return X, Y, D, C, df_info

# ==========================================================
# 2) MÉTRICAS — (mantive como você mandou)
# ==========================================================
def calculate_evm(ref, test):
    ref = np.asarray(ref); test = np.asarray(test)
    rc = ref[:,0] + 1j*ref[:,1]
    tc = test[:,0] + 1j*test[:,1]
    mean_power = np.mean(np.abs(rc)**2)
    if mean_power == 0: return float("inf"), float("-inf")
    evm = np.sqrt(np.mean(np.abs(tc-rc)**2) / mean_power)
    return float(evm*100), float(20*np.log10(max(evm,1e-12)))

def calculate_snr(ref, test):
    ref = np.asarray(ref); test = np.asarray(test)
    rc = ref[:,0] + 1j*ref[:,1]
    tc = test[:,0] + 1j*test[:,1]
    sp = np.mean(np.abs(rc)**2)
    npow = np.mean(np.abs(rc-tc)**2)
    if npow == 0: return float("inf")
    return float(10*np.log10(max(sp/npow,1e-12)))

def calculate_noise_power(sent, recv):
    noise = recv - sent
    return float(np.mean(np.sum(noise**2, axis=1)))

def generate_awgn_output(x_in, target_noise_power):
    sigma = np.sqrt(target_noise_power / 2.0)
    noise = np.random.normal(0, sigma, x_in.shape).astype(np.float32)
    return x_in + noise

def acf_mse(noise_real, noise_pred, nlags=40):
    import statsmodels.tsa.stattools as tsa
    r_i = tsa.acf(noise_real[:,0], nlags=nlags, fft=True)
    p_i = tsa.acf(noise_pred[:,0], nlags=nlags, fft=True)
    r_q = tsa.acf(noise_real[:,1], nlags=nlags, fft=True)
    p_q = tsa.acf(noise_pred[:,1], nlags=nlags, fft=True)
    return float(0.5*(np.mean((r_i-p_i)**2)+np.mean((r_q-p_q)**2)))

def psd_mse(noise_real, noise_pred, fs=1.0, nperseg=1024):
    from scipy.signal import welch
    f, r_i = welch(noise_real[:,0], fs=fs, nperseg=nperseg)
    f, p_i = welch(noise_pred[:,0], fs=fs, nperseg=nperseg)
    f, r_q = welch(noise_real[:,1], fs=fs, nperseg=nperseg)
    f, p_q = welch(noise_pred[:,1], fs=fs, nperseg=nperseg)
    eps = 1e-20
    mi = np.mean((np.log10(r_i+eps)-np.log10(p_i+eps))**2)
    mq = np.mean((np.log10(r_q+eps)-np.log10(p_q+eps))**2)
    return float(0.5*(mi+mq))

def jsd_kde_2d(real, pred, grid_res=100, bw=0.12):
    real = np.asarray(real); pred = np.asarray(pred)
    mins = np.minimum(real.min(axis=0), pred.min(axis=0))
    maxs = np.maximum(real.max(axis=0), pred.max(axis=0))
    xs = np.linspace(mins[0], maxs[0], grid_res)
    ys = np.linspace(mins[1], maxs[1], grid_res)
    Xg, Yg = np.meshgrid(xs, ys)
    pos = np.stack([Xg.ravel(), Yg.ravel()], axis=1)

    def kde(data):
        d = data.astype(np.float32)
        out = np.zeros((pos.shape[0],), dtype=np.float64)
        bs = 3000
        inv = 1.0/(2*np.pi*bw*bw)
        for i in range(0, d.shape[0], bs):
            b = d[i:i+bs]
            dx = (b[:,0,None]-pos[:,0])/bw
            dy = (b[:,1,None]-pos[:,1])/bw
            out += np.sum(np.exp(-0.5*(dx*dx+dy*dy)), axis=0)
        out = out / max(1, d.shape[0])
        out = out * inv
        s = out.sum()
        if s > 0: out /= s
        return out.astype(np.float64)

    P = kde(real)
    Q = kde(pred)
    M = 0.5*(P+Q)
    eps = 1e-12
    jsd = 0.5*np.sum(P*np.log2((P+eps)/(M+eps))) + 0.5*np.sum(Q*np.log2((Q+eps)/(M+eps)))
    return float(jsd)

# ==========================================================
# 3) CAMADAS CUSTOM (sem decorator; definidas 1x)
# ==========================================================
if "Sampling" not in globals():
    class Sampling(layers.Layer):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
        def call(self, inputs):
            z_mean, z_log_var = inputs
            eps = tf.random.normal(tf.shape(z_mean))
            return z_mean + tf.exp(0.5*z_log_var)*eps
        def get_config(self):
            return super().get_config()

if "HeteroscedasticVAELossLayer" not in globals():
    class HeteroscedasticVAELossLayer(layers.Layer):
        def __init__(self, beta=1.0, free_bits=0.0, **kwargs):
            super().__init__(**kwargs)
            self.beta_init = float(beta)
            self.free_bits = float(free_bits)
            self.beta = tf.Variable(self.beta_init, trainable=False, dtype=tf.float32)
            self.recon_loss_tracker = tf.keras.metrics.Mean(name="recon_loss")
            self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

        def call(self, inputs):
            y_true, y_pred_params, z_mean, z_log_var = inputs

            y_mean = y_pred_params[:, :2]
            y_log_var = y_pred_params[:, 2:]
            y_log_var = tf.clip_by_value(y_log_var, -6.0, 2.0)
            y_var = tf.exp(y_log_var) + 1e-6

            nll = 0.5 * tf.reduce_sum(
                y_log_var + tf.square(y_true - y_mean) / y_var + tf.math.log(2.0*np.pi),
                axis=-1
            )
            recon = tf.reduce_mean(nll)

            kl_per_sample = -0.5 * tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=-1
            )
            fb = tf.cast(self.free_bits, kl_per_sample.dtype)
            kl_fb = tf.maximum(kl_per_sample - fb, 0.0)
            kl = tf.reduce_mean(kl_fb)

            total = recon + self.beta * tf.minimum(kl, 100.0)
            self.add_loss(total)
            self.recon_loss_tracker.update_state(recon)
            self.kl_loss_tracker.update_state(tf.reduce_mean(kl_per_sample))
            return y_mean

        @property
        def metrics(self):
            return [self.recon_loss_tracker, self.kl_loss_tracker]

        def get_config(self):
            cfg = super().get_config()
            cfg.update({"beta": self.beta_init, "free_bits": self.free_bits})
            return cfg

# ==========================================================
# 4) CARREGAR MODELOS (.keras) — robusto p/ FULL / PRIOR / DECODER
#     (Opção A exige prior + decoder; pode estar tudo no full, ou separado)
# ==========================================================
def list_keras(d: Path):
    return sorted([p for p in d.glob("*.keras") if p.is_file()]) if (d and d.exists()) else []

cand = []
cand += list_keras(MODELS_DIR)
cand += [p for p in list_keras(RUN_DIR) if p not in cand]

if not cand:
    raise FileNotFoundError(f"Não encontrei .keras em {MODELS_DIR} nem em {RUN_DIR}")

print("📦 Candidatos .keras:")
for p in cand:
    print("   -", p.name)

def _score(p: Path, kind: str):
    n = p.name.lower()
    s = 0
    if kind == "decoder":
        if "decoder" in n: s += 100
        if "production" in n: s += 5
    elif kind == "prior":
        if "prior" in n: s += 100
        if "prior_net" in n: s += 10
        if "production" in n: s += 5
    elif kind == "full":
        if ("cvae" in n) or ("vae" in n): s += 50
        if "decoder" in n: s -= 80
        if "prior" in n: s -= 80
        if "inference" in n: s -= 20
        if "multi_current" in n: s += 5
    return s

decoder_path = sorted(cand, key=lambda p: _score(p, "decoder"), reverse=True)[0]
prior_path   = sorted(cand, key=lambda p: _score(p, "prior"), reverse=True)[0]
full_path    = sorted(cand, key=lambda p: _score(p, "full"), reverse=True)[0]

# só aceita como "encontrado" se score > 0
decoder_path = decoder_path if _score(decoder_path, "decoder") > 0 else None
prior_path   = prior_path   if _score(prior_path,   "prior")   > 0 else None
full_path    = full_path    if _score(full_path,    "full")    > 0 else None

print(f"\n✅ FULL:   {full_path.name if full_path else None}")
print(f"✅ PRIOR:  {prior_path.name if prior_path else None}")
print(f"✅ DECODER:{decoder_path.name if decoder_path else None}")

custom_objects = {
    "Sampling": Sampling,
    "HeteroscedasticVAELossLayer": HeteroscedasticVAELossLayer,
}

def _load(path: Path):
    return tf.keras.models.load_model(str(path), custom_objects=custom_objects, compile=False)

vae_full = _load(full_path) if full_path else None
prior_model = None
decoder_model = None

def _has_layer(model, name):
    try:
        model.get_layer(name)
        return True
    except Exception:
        return False

# Caso 1: full contém prior_net e decoder
if vae_full is not None and _has_layer(vae_full, "prior_net") and _has_layer(vae_full, "decoder"):
    prior_model  = vae_full.get_layer("prior_net")
    decoder_model = vae_full.get_layer("decoder")
    print("✓ Usando prior_net + decoder DE DENTRO do FULL.")
else:
    # Caso 2: modelos separados (prior e decoder salvos em arquivos)
    if prior_path is None:
        raise ValueError("Não encontrei arquivo do prior (nome contendo 'prior'). Para Opção A precisa do prior.")
    if decoder_path is None:
        raise ValueError("Não encontrei arquivo do decoder (nome contendo 'decoder'). Para Opção A precisa do decoder.")

    prior_loaded = _load(prior_path)
    decoder_loaded = _load(decoder_path)

    # prior pode ser um Model([x,d,c]) -> [mu, logvar], ou um full contendo z_mean_p layers
    if len(prior_loaded.outputs) == 2 and len(prior_loaded.inputs) >= 3:
        prior_model = prior_loaded
    else:
        # tenta extrair via layers z_mean_p/z_log_var_p
        if _has_layer(prior_loaded, "z_mean_p") and _has_layer(prior_loaded, "z_log_var_p"):
            z_mean_p = prior_loaded.get_layer("z_mean_p").output
            z_log_var_p = prior_loaded.get_layer("z_log_var_p").output
            prior_model = tf.keras.Model(prior_loaded.inputs[:3], [z_mean_p, z_log_var_p], name="prior_conditional")
        else:
            raise ValueError("O prior carregado não parece produzir (z_mean_p, z_log_var_p).")

    decoder_model = decoder_loaded
    print("✓ Usando PRIOR e DECODER de arquivos separados.")

# ==========================================================
# 5) INFERÊNCIA (prior condicional + decoder) — Opção A
# ==========================================================
def create_inference_model_from_condprior(prior, dec,
                                         clip_zlogvar=(-10.0, 10.0),
                                         clip_ylogvar=(-6.0, 2.0)):
    x_in = layers.Input(shape=(2,), name="x_input")
    d_in = layers.Input(shape=(1,), name="distance_input")
    c_in = layers.Input(shape=(1,), name="current_input")

    z_mean_p, z_log_var_p = prior([x_in, d_in, c_in], training=False)
    z_log_var_p = tf.clip_by_value(z_log_var_p, clip_zlogvar[0], clip_zlogvar[1])

    eps_z = tf.random.normal(tf.shape(z_mean_p))
    z = z_mean_p + tf.exp(0.5 * z_log_var_p) * eps_z

    cond = layers.Concatenate()([x_in, d_in, c_in])  # (N,4)
    out_params = dec([z, cond], training=False)

    y_mean = out_params[:, :2]
    y_log_var = tf.clip_by_value(out_params[:, 2:], clip_ylogvar[0], clip_ylogvar[1])

    eps_y = tf.random.normal(tf.shape(y_mean))
    y = y_mean + tf.exp(0.5 * y_log_var) * eps_y

    return tf.keras.Model([x_in, d_in, c_in], y, name="cvae_inference_condprior")

inference_model = create_inference_model_from_condprior(prior_model, decoder_model)
print("\n✅ inference_model pronto:", inference_model.name)

# ==========================================================
# 6) DATASET ROOT (do state ou env/default) — pronto para gerar dados/plots
# ==========================================================
DATASET_ROOT = Path(state.get("dataset_root", os.environ.get("DATASET_ROOT", "/workspace/2026/dataset_fullsquare_organized")))
print(f"📁 DATASET_ROOT = {DATASET_ROOT}")

# (a partir daqui, o resto do teu bloco 3 segue igual: load_all_datasets_auto, normalização, split, predict, etc.)